# Liberman synapse comparison — classical (linear + trees)

Liberman **scenario C** only. Runs configuration panel **T1, T3–T6** (RF + XGB each) and **L7–L8** (OLS). Strain is omitted (not significant at any test-tone frequency). Wide tree/OLS stage-2 features use **50/60/70/80 dB** pivoted columns only (stage 1 noise clf too). Long stage-2 (T1, T3) uses all SPL rows.

Data lineage: [`abr_wide_long_comparison.ipynb`](abr_wide_long_comparison.ipynb). Exports: `figures/cache/liberman_classical_comparison.parquet`, `liberman_best_tree_config.json`. No plots here — synthesis elsewhere.

| ID | Spec |
|----|------|
| T1 | Long, no label |
| T3 | Long, noise_pred |
| T4 | Wide, no label |
| T5 | Wide, noise_pred |
| T6 | Wide, true noise_cat (animal-level) |
| L7 | Wide OLS amp 80 dB |
| L8 | Wide OLS full + noise_pred |

In [1]:
import json
import importlib
from pathlib import Path
from dataclasses import replace

import pandas as pd
from IPython.display import display

import utils.liberman_classical as lc
import utils.nn_stage2_data as nn2d
import utils.nn_stage2 as nn2

importlib.reload(nn2d)
importlib.reload(nn2)
importlib.reload(lc)

from utils.liberman_classical import (
    EXCLUDE_S2_EXTRA,
    animal_noise_series,
    attach_animal_noise_cat,
    derive_comparisons,
    diagnose_wide_noise_lift,
    export_artifacts,
    export_noise_diagnostic,
    export_t5_stage2_hp,
    liberman_feature_lists,
    run_config_panel,
)
from utils.nn_stage2 import RunConfig, run_two_stage_long_nn
from utils.nn_stage2_data import (
    STAGE_SPL_LEVELS,
    load_nn_stage2_data,
    splits_for_long_stage2,
)

In [2]:
data = load_nn_stage2_data(join_io_features=False)
sp = splits_for_long_stage2(data)

lib_tr = sp["lib_train"].copy()
lib_te = sp["lib_test"].copy()
lib_long_tr = sp["lib_long_train"].copy()
lib_long_te = sp["lib_long_test"].copy()

_animal_noise = animal_noise_series(data.orig_lib)
lib_tr = attach_animal_noise_cat(lib_tr, _animal_noise)
lib_te = attach_animal_noise_cat(lib_te, _animal_noise)
lib_long_tr = attach_animal_noise_cat(lib_long_tr, _animal_noise)
lib_long_te = attach_animal_noise_cat(lib_long_te, _animal_noise)

feats = liberman_feature_lists(data.reformatted_orig, data.common_cols)

assert lib_tr["noise_cat"].isin([0, 1]).all()
assert lib_tr.groupby("animal_id")["noise_cat"].nunique().max() == 1

print(f"Wide tree SPL levels: {list(STAGE_SPL_LEVELS)} dB")
print(
    f"Liberman wide train/test: {len(lib_tr)} / {len(lib_te)} rows | "
    f"long train/test (all SPL): {len(lib_long_tr)} / {len(lib_long_te)}"
)
print(f"Synapse wide num/log: {len(feats['syn_num'])} / {len(feats['syn_log'])}")
print(f"Excluded extras present in syn_num: {set(feats['syn_num']) & EXCLUDE_S2_EXTRA}")

Wide tree SPL levels: [50, 60, 70, 80] dB
Liberman wide train/test: 491 / 125 rows | long train/test (all SPL): 9751 / 2436
Synapse wide num/log: 12 / 29
Excluded extras present in syn_num: set()


In [3]:
results = run_config_panel(
    lib_tr, lib_te, lib_long_tr, lib_long_te, feats, verbose=True
)
assert len(results) == 12, f"expected 12 rows, got {len(results)}"
display(results.sort_values(["config_id", "model"]))


=== T1 RF (long, noise=none) ===
  [T1-RF]  S1 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse reg — CV R²: 0.169, test R² (animal×freq): 0.385, RMSE: 2.876

=== T1 XGB (long, noise=none) ===
  [T1-XGB]  S1/S2 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse XGB — CV R²: 0.170, test R² (animal×freq): 0.381, RMSE: 2.884

=== T3 RF (long, noise=predicted) ===
  [T3-RF]  S1 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse reg — CV R²: 0.437, test R² (animal×freq): 0.590, RMSE: 2.346

=== T3 XGB (long, noise=predicted) ===
  [T3-XGB]  S1/S2 wi

,config_id,model,format,noise_label,r2_test,rmse_test
10,L7,OLS,wide,none,0.228920,3.090075
11,L8,OLS,wide,predicted,0.424054,2.670609
0,T1,RF,long,none,0.384623,2.875946
1,T1,XGB,long,none,0.381239,2.883842
2,T3,RF,long,predicted,0.590469,2.346137
3,T3,XGB,long,predicted,0.552502,2.452479
4,T4,RF,wide,none,0.490445,2.511971
5,T4,XGB,wide,none,0.528145,2.417261
6,T5,RF,wide,predicted,0.643112,2.102253
7,T5,XGB,wide,predicted,0.603899,2.214738


In [4]:
# Export T5 wide RF/XGB best_params for Section 5.3 scenario C (skip if JSON exists)
_hp_out = Path("figures/cache/stage2_best_hp/liberman_t5_sklearn.json")
if not _hp_out.is_file():
    export_t5_stage2_hp(lib_tr, lib_te, feats, out_path=_hp_out)
    print("Wrote", _hp_out)
else:
    print("Skip T5 HP export — already present:", _hp_out)

Wrote figures/cache/stage2_best_hp/liberman_t5_sklearn.json


In [5]:
comparisons = derive_comparisons(
    results,
    wide_train=lib_tr,
    wide_test=lib_te,
    long_train=lib_long_tr,
    long_test=lib_long_te,
    feats=feats,
)
display(comparisons)

for model in ("RF", "XGB"):
    q3 = comparisons[
        (comparisons["question"] == "Q3_format") & (comparisons["model"] == model)
    ]
    if not q3.empty:
        d = q3.iloc[0]["delta_r2"]
        fmt = "wide" if d > 0 else "long"
        print(f"  Q3 best format ({model}): {fmt} (delta_r2 T4-T1 = {d:.3f})")

,question,model,config_a,config_b,r2_a,r2_b,delta_r2,rmse_a,rmse_b,delta_rmse,...,mean_sq_err_b,mean_sq_err_diff_a_minus_b,f_stat,f_pvalue,variances_unequal,t_test,t_stat,t_pvalue,significant_at_alpha,alpha
0,Q3_format,RF,T1,T4,0.384623,0.490445,0.105823,2.875946,2.511971,-0.363975,...,6.309998,1.961067,1.598107,0.009507,True,welch,1.330329,0.184697,False,0.05
1,Q1_noise,RF,T4,T5,0.490445,0.643112,0.152667,2.511971,2.102253,-0.409718,...,4.419467,1.890531,1.539357,0.016953,True,welch,1.609484,0.108840,False,0.05
2,Q1_noise_long,RF,T1,T3,0.384623,0.590469,0.205846,2.875946,2.346137,-0.529809,...,5.504359,2.766706,1.655970,0.005303,True,welch,1.889599,0.060049,False,0.05
3,Q2_oracle,RF,T5,T6,0.643112,0.650047,0.006934,2.102253,2.081729,-0.020524,...,4.333597,0.085870,1.021006,0.908042,False,paired,0.799917,0.425289,False,0.05
4,Q3_format,XGB,T1,T4,0.381239,0.528145,0.146906,2.883842,2.417261,-0.466580,...,5.843152,2.473391,1.749436,0.002016,True,welch,1.711455,0.088340,False,0.05
5,Q1_noise,XGB,T4,T5,0.528145,0.603899,0.075754,2.417261,2.214738,-0.202523,...,4.905064,0.938088,1.062069,0.737939,False,paired,1.717218,0.088436,False,0.05
6,Q1_noise_long,XGB,T1,T3,0.381239,0.552502,0.171263,2.883842,2.452479,-0.431363,...,6.014654,2.301889,1.354259,0.092599,False,paired,3.442363,0.000787,True,0.05
7,Q2_oracle,XGB,T5,T6,0.603899,0.634151,0.030252,2.214738,2.128483,-0.086255,...,4.530438,0.374626,1.151604,0.433030,False,paired,1.514665,0.132402,False,0.05


  Q3 best format (RF): wide (delta_r2 T4-T1 = 0.106)
  Q3 best format (XGB): wide (delta_r2 T4-T1 = 0.147)


## Wide noise diagnostic (T4/T5/T6)

Stage-1 test errors vs stage-2 under-using ``noise_preds``. Also audits **train** ``noise_preds`` vs ``noise_cat`` (T5 vs T6).

In [6]:
cache_dir = Path("figures/cache")
for _model in ("RF", "XGB"):
    _anim, _train, _summary = diagnose_wide_noise_lift(
        lib_tr, lib_te, feats, model=_model
    )
    display(_anim.sort_values("pred_correct"))
    display(_train.sort_values("pred_correct"))
    print(json.dumps(_summary, indent=2))
    export_noise_diagnostic(
        _anim, _summary, cache_dir, model=_model, train_animal_tbl=_train
    )

,animal_id,noise_cat,noise_pred,noise_prob,pred_correct,n_rows,synapses,mse_T4,mse_T5,mse_T6,delta_mse_T5_minus_T4,delta_mse_T6_minus_T5
3,WPZ113,1,0,0.366657,False,6,14.898821,1.459881,2.535214,1.076992,1.075333,-1.458222
0,WPZ100,0,0,0.514810,True,6,15.892142,5.430388,1.345499,1.402921,-4.084889,0.057422
18,WPZ91,0,0,0.387516,True,6,13.842553,4.181667,2.280362,2.212812,-1.901305,-0.067550
17,WPZ83,0,0,0.442879,True,6,15.269309,1.353582,1.017169,0.884737,-0.336413,-0.132432
16,WPZ67,0,0,0.373788,True,6,14.620126,8.152160,10.001281,9.972241,1.849121,-0.029040
15,WPZ62,1,1,0.591207,True,6,15.440508,4.178034,2.189147,2.583526,-1.988887,0.394379
14,WPZ51,1,1,0.676019,True,6,15.213716,6.016625,6.388002,5.697822,0.371377,-0.690180
13,WPZ179,1,1,0.744833,True,5,12.780952,14.702872,9.129901,9.779841,-5.572971,0.649941
12,WPZ171,1,1,0.622090,True,6,15.559913,5.973716,4.929929,4.511052,-1.043787,-0.418878
11,WPZ165,0,0,0.403734,True,6,14.892857,2.862121,0.829108,0.809635,-2.033014,-0.019472


,animal_id,noise_cat,noise_pred,n_rows,pred_correct
10,WPZ123,1,0,6,False
59,WPZ60,0,1,6,False
8,WPZ117,1,0,6,False
61,WPZ66,1,0,6,False
32,WPZ155,1,0,5,False
...,...,...,...,...,...
25,WPZ144,1,1,5,True
24,WPZ143,1,1,6,True
23,WPZ142,0,0,6,True
42,WPZ169,1,1,6,True


{
  "model": "RF",
  "stage1": {
    "n_test_animals": 21,
    "n_correct": 20,
    "n_wrong": 1,
    "accuracy": 0.9523809523809523,
    "wrong_animal_ids": [
      "WPZ113"
    ],
    "threshold": 0.5557642115606013,
    "test_auc": 0.9090909090909091,
    "non_test_train_animal_accuracy": 0.9404761904761905
  },
  "train_noise_labels": {
    "n_train_animals": 84,
    "n_train_wide_rows": 491,
    "noise_preds_nan_rows": 0,
    "row_match_rate": 0.9409368635437881,
    "animal_match_rate": 0.9404761904761905,
    "n_mismatched_animals": 5,
    "mismatched_animal_ids": [
      "WPZ117",
      "WPZ123",
      "WPZ155",
      "WPZ60",
      "WPZ66"
    ]
  },
  "n_test_wide_rows": 125,
  "n_rows_correct_animals": 119,
  "n_rows_wrong_animals": 6,
  "r2_all_rows": {
    "T4": 0.49044530875565284,
    "T5": 0.643112386374932,
    "T5_oracle_train_labels": 0.6439420553505474,
    "T6": 0.6500467152002691,
    "T5_oracle_test_labels": 0.6436933169162291
  },
  "r2_correct_animals_only": {


,animal_id,noise_cat,noise_pred,noise_prob,pred_correct,n_rows,synapses,mse_T4,mse_T5,mse_T6,delta_mse_T5_minus_T4,delta_mse_T6_minus_T5
3,WPZ113,1,0,0.366657,False,6,14.898821,1.503706,2.363421,1.284361,0.859715,-1.079059
0,WPZ100,0,0,0.514810,True,6,15.892142,5.640720,2.011380,2.846975,-3.629340,0.835594
18,WPZ91,0,0,0.387516,True,6,13.842553,5.144118,3.033083,2.036408,-2.111035,-0.996674
17,WPZ83,0,0,0.442879,True,6,15.269309,1.694496,1.387125,1.232347,-0.307370,-0.154778
16,WPZ67,0,0,0.373788,True,6,14.620126,7.140979,9.359175,8.661110,2.218197,-0.698066
15,WPZ62,1,1,0.591207,True,6,15.440508,4.915539,3.723422,2.930282,-1.192117,-0.793140
14,WPZ51,1,1,0.676019,True,6,15.213716,3.593576,6.296463,7.960352,2.702887,1.663889
13,WPZ179,1,1,0.744833,True,5,12.780952,16.194717,12.186869,10.892418,-4.007848,-1.294451
12,WPZ171,1,1,0.622090,True,6,15.559913,3.796985,5.820254,2.205052,2.023269,-3.615202
11,WPZ165,0,0,0.403734,True,6,14.892857,2.456102,0.959381,1.248799,-1.496720,0.289418


,animal_id,noise_cat,noise_pred,n_rows,pred_correct
10,WPZ123,1,0,6,False
59,WPZ60,0,1,6,False
8,WPZ117,1,0,6,False
61,WPZ66,1,0,6,False
32,WPZ155,1,0,5,False
...,...,...,...,...,...
25,WPZ144,1,1,5,True
24,WPZ143,1,1,6,True
23,WPZ142,0,0,6,True
42,WPZ169,1,1,6,True


{
  "model": "XGB",
  "stage1": {
    "n_test_animals": 21,
    "n_correct": 20,
    "n_wrong": 1,
    "accuracy": 0.9523809523809523,
    "wrong_animal_ids": [
      "WPZ113"
    ],
    "threshold": 0.5557642115606013,
    "test_auc": 0.9090909090909091,
    "non_test_train_animal_accuracy": 0.9404761904761905
  },
  "train_noise_labels": {
    "n_train_animals": 84,
    "n_train_wide_rows": 491,
    "noise_preds_nan_rows": 0,
    "row_match_rate": 0.9409368635437881,
    "animal_match_rate": 0.9404761904761905,
    "n_mismatched_animals": 5,
    "mismatched_animal_ids": [
      "WPZ117",
      "WPZ123",
      "WPZ155",
      "WPZ60",
      "WPZ66"
    ]
  },
  "n_test_wide_rows": 125,
  "n_rows_correct_animals": 119,
  "n_rows_wrong_animals": 6,
  "r2_all_rows": {
    "T4": 0.5281447727351727,
    "T5": 0.6038986942876202,
    "T5_oracle_train_labels": 0.6272294256140584,
    "T6": 0.6341510695108128,
    "T5_oracle_test_labels": 0.6022490015493598
  },
  "r2_correct_animals_only": {

## Export Colab GPU pack (NN HP tuning)

Preprocessed long-format tensors for [`abr_nn_stage2_colab.ipynb`](abr_nn_stage2_colab.ipynb): tabular features (fitted scaler on Train rows), Wave-I / full-wave arrays, Stage-1 `noise_preds`, and train / validate / test splits. Upload `figures/cache/nn_colab_liberman/` to Google Drive before running the Colab notebook.

In [7]:
from utils.nn_colab_export import DEFAULT_OUT, export_liberman_nn_colab_pack

colab_pack_dir = export_liberman_nn_colab_pack(DEFAULT_OUT)
print(f"Upload to Colab: {colab_pack_dir.resolve()}")

           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909
Exported Colab pack → /Users/nowaki027/MSDS/Practicum/figures/cache/nn_colab_liberman
  train 7869 rows, 68 animals | validate 1882 | test 2436
  tabular dim=13, wave_full_len=201
Upload to Colab: /Users/nowaki027/MSDS/Practicum/figures/cache/nn_colab_liberman


## NN Stage 2 (long) — Liberman only

Three NN variants from `abr_nn_stage2.ipynb`, trained/evaluated on the **Liberman** split only:

- `mlp`
- `cnn` (Wave I)
- `cnn_full` (full 0–8 ms segment)

All use **noise_pred** (stage 1) and **exclude strain**.

In [8]:
nn_data = replace(data, long_num=[c for c in data.long_num if c != "strain_binary"])
cfg = RunConfig(full_wave_ref="lib")

nn_rows = []
for name, mode in [("N1", "mlp"), ("N2", "cnn"), ("N3", "cnn_full")]:
    r2, rmse = run_two_stage_long_nn(
        lib_tr,
        lib_te,
        lib_long_tr,
        lib_long_te,
        label=f"{name}-{mode}",
        data=nn_data,
        noise_num=data.noise_num_lib,
        noise_log=data.noise_log_lib,
        mode=mode,
        cfg=cfg,
        verbose=True,
    )
    nn_rows.append(
        {
            "config_id": name,
            "model": mode,
            "format": "long",
            "noise_label": "predicted",
            "r2_test": float(r2),
            "rmse_test": float(rmse),
        }
    )

nn_results = pd.DataFrame(nn_rows)
display(nn_results)

  [N1-mlp]  S1 fit/val wide: 397/94 | long train: 9751 rows | Stage2=mlp
           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909
           Stage2 rows — train: 7869 | validate: 1882 | animals train/val: 68/16


RuntimeError: Tensor for argument input is on cpu but expected on mps

In [9]:
cache_dir = Path("figures/cache")
pq_path, json_path, comp_path = export_artifacts(results, comparisons, cache_dir)
print(f"Wrote {pq_path}")
print(f"Wrote {comp_path}")
print(f"Wrote {json_path}")

Wrote figures/cache/liberman_classical_comparison.parquet
Wrote figures/cache/liberman_classical_comparisons.parquet
Wrote figures/cache/liberman_best_tree_config.json
